In [ ]:
import numpy as np
import cv2

In [ ]:
def compute_laplacian_variance(image):
    """Wariancja Laplacian'a - czułość na rozmycie"""
    laplacian = cv2.Laplacian(image, cv2.CV_64F)
    variance = laplacian.var()
    return variance

def compute_power_spectrum_ratio(image):
    """Ratio mocy wysokich vs niskich częstotliwości"""
    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)
    
    h, w = magnitude.shape
    center_h, center_w = h // 2, w // 2
    
    # Niskie częstotliwości (środek)
    low_freq = np.mean(magnitude[center_h-20:center_h+20, center_w-20:center_w+20])
    
    # Wysokie częstotliwości (rogi)
    high_freq = (np.mean(magnitude[:30, :30]) + 
                 np.mean(magnitude[-30:, -30:]) + 
                 np.mean(magnitude[:30, -30:]) + 
                 np.mean(magnitude[-30:, :30])) / 4
    
    # Ratio - im większy, tym mniej rozmycia
    ratio = high_freq / (low_freq + 1e-6)
    return ratio

def compute_entropy(image):
    """Entropia Shannon'a histogramu"""
    hist, _ = np.histogram(image, bins=256, range=(0, 256))
    hist = hist / hist.sum()  # Normalizacja
    hist = hist[hist > 0]  # Unikaj log(0)
    entropy = -np.sum(hist * np.log2(hist))
    return entropy

def compute_harris_corners(image):
    """Ilość narożników Harris'a"""
    gray = image.astype(np.float32)
    corners = cv2.cornerHarris(gray, 2, 3, 0.04)
    corner_count = np.sum(corners > 0.01 * corners.max())
    return corner_count

def compute_gradient_energy(image):
    """Energia gradientu - suma kwadratów gradientów"""
    sobelx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    gradient_energy = np.sum(sobelx**2 + sobely**2)
    return gradient_energy

def compute_histogram_properties(image):
    """Skośność i kurtoza histogramu"""
    hist, _ = np.histogram(image, bins=256, range=(0, 256))
    
    # Średnia i std
    mean = np.average(np.arange(256), weights=hist)
    std = np.sqrt(np.average((np.arange(256) - mean)**2, weights=hist))
    
    # Skośność
    skewness = np.average((np.arange(256) - mean)**3, weights=hist) / (std**3 + 1e-6)
    
    # Kurtoza
    kurtosis = np.average((np.arange(256) - mean)**4, weights=hist) / (std**4 + 1e-6)
    
    return skewness, kurtosis

In [5]:
img = cv2.imread('../data/gopro_deblur/blur/images/000001.png')

In [7]:
compute_laplacian_variance(img)

np.float64(15.001649282037343)

In [ ]:
img = cv2.imread('../data/gopro_deblur/r/imddasz 00001.png')